In [ ]:
import pandas as pd
import json
import os
import math
import numpy as np
import librosa, librosa.display
import kagglehub
from sklearn.model_selection import train_test_split
import sklearn.preprocessing as skp
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

     

In [ ]:
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
df = pd.read_csv(path + "/Data/features_3_sec.csv")
df.head(10)

In [ ]:
print("Dataset has",df.shape)
print("Count of Positive and Negative samples")
df.label.value_counts().reset_index()

In [ ]:
df.label.unique()

In [ ]:
print("Columns with NA values are",list(df.columns[df.isnull().any()]))

In [ ]:

df.dtypes

In [ ]:
df = df.drop(labels="filename", axis=1)

In [ ]:

df.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
x = df[["label", "tempo"]]

fig, ax = plt.subplots(figsize=(16, 8));
sns.boxplot(x = "label", y = "tempo", data = x, palette = 'husl');

plt.title('BPM Boxplot for Genres', fontsize = 20)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 10);
plt.xlabel("Genre", fontsize = 15)
plt.ylabel("BPM", fontsize = 15)
plt.savefig("BPM_Boxplot.png")
     


In [ ]:
data = df.iloc[0:, 1:]
y = data['label']
X = data.loc[:, data.columns != 'label']

# normalize
cols = X.columns
min_max_scaler = skp.MinMaxScaler()
np_scaled = min_max_scaler.fit_transform(X)
X = pd.DataFrame(np_scaled, columns = cols)

# Top 2 pca components
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X)
principalDf = pd.DataFrame(data = principalComponents, columns = ['pc1', 'pc2'])

# concatenate with target label
finalDf = pd.concat([principalDf, y], axis = 1)

plt.figure(figsize = (16, 9))
sns.scatterplot(x = "pc1", y = "pc2", data = finalDf, hue = "label", alpha = 0.7, s = 100);

plt.title('PCA on Genres', fontsize = 20)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 10);
plt.xlabel("Principal Component 1", fontsize = 15)
plt.ylabel("Principal Component 2", fontsize = 15)
plt.savefig("PCA_Scattert.png")
     


In [ ]:
DATASET_PATH = path + "/Data/genres_original"
JSON_PATH = "../../data/gtzan/data_10.json"
SAMPLE_RATE = 22050
TRACK_DURATION = 30 # measured in seconds
SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION

### MFCC (Mel-frequency cepstral coefficients)

MFCC (Mel-frequency cepstral coefficients) are features commonly used in audio processing and speech recognition to represent the characteristics of sound. They summarize the power spectrum of an audio signal on a nonlinear mel scale, which mimics human hearing. Here's the process:

Pre-emphasis: Amplifies high frequencies to balance the spectrum.
Frame the signal: Divide the signal into short overlapping frames.
Fourier Transform: Converts time-domain signal to frequency-domain.
Mel Filter Bank: Apply filters spaced on the mel scale.
Logarithm: Compresses the signal range.
Discrete Cosine Transform (DCT): Reduces dimensionality by creating a compact representation.

MFCCs capture timbral information, often used for audio classification or speech recognition.

In [ ]:
def save_mfcc(dataset_path, json_path, num_mfcc=13, n_fft=2048, hop_length=512, num_segments=5):
    """Extracts MFCCs from music dataset and saves them into a json file along witgh genre labels.
        :param dataset_path (str): Path to dataset
        :param json_path (str): Path to json file used to save MFCCs
        :param num_mfcc (int): Number of coefficients to extract
        :param n_fft (int): Interval we consider to apply FFT. Measured in # of samples
        :param hop_length (int): Sliding window for FFT. Measured in # of samples
        :param: num_segments (int): Number of segments we want to divide sample tracks into
        :return:
        """

    # dictionary to store mapping, labels, and MFCCs
    data = {
        "mapping": [],
        "labels": [],
        "mfcc": []
    }

    samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    num_mfcc_vectors_per_segment = math.ceil(samples_per_segment / hop_length)

    # loop through all genre sub-folder
    for i, (dirpath, dirnames, filenames) in enumerate(os.walk(dataset_path)):

        # ensure we're processing a genre sub-folder level
        if dirpath is not dataset_path:

            # save genre label (i.e., sub-folder name) in the mapping
            semantic_label = dirpath.split("/")[-1]
            data["mapping"].append(semantic_label)
            print("\nProcessing: {}".format(semantic_label))

            # process all audio files in genre sub-dir
            for f in filenames:

                # load audio file

                file_path = os.path.join(dirpath, f)
            
                if file_path != path + '/Data/genres_original/jazz/jazz.00054.wav':

                    signal, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
                
                
                    # process all segments of audio file
                    for d in range(num_segments):

                        # calculate start and finish sample for current segment
                        start = samples_per_segment * d
                        finish = start + samples_per_segment

                        # extract mfcc
                        mfcc = librosa.feature.mfcc(y=signal[start:finish], sr=sample_rate, n_mfcc=num_mfcc, n_fft=n_fft, hop_length=hop_length)
                        mfcc = mfcc.T

                        # store only mfcc feature with expected number of vectors
                        if len(mfcc) == num_mfcc_vectors_per_segment:
                            data["mfcc"].append(mfcc.tolist())
                            data["labels"].append(i-1)
                            print("{}, segment:{}".format(file_path, d+1))

    # save MFCCs to json file
    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)

In [ ]:
save_mfcc(DATASET_PATH, JSON_PATH)

In [ ]:
# path to json
DATA_PATH = JSON_PATH

In [ ]:
def load_data(data_path):

    with open(data_path, "r") as f:
        data = json.load(f)

    # convert lists to numpy arrays
    X = np.array(data["mfcc"])
    y = np.array(data["labels"])

    print("Data succesfully loaded!")

    return  X, y
     

In [ ]:
y.shape

In [ ]:
# load data
X, y = load_data(DATA_PATH)
     

In [ ]:
X.shape

In [ ]:
 #Reshape X to 2D for XGBoost compatibility
X_reshaped = X.reshape(X.shape[0], -1)



In [ ]:
X_reshaped.shape

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

# Initialize XGBoost classifier
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Train the model
xgb_clf.fit(X_train, y_train)

# Predict probabilities
y_pred_prob = xgb_clf.predict_proba(X_test)

# Optional: Get predicted classes
y_pred_class = np.argmax(y_pred_prob, axis=1)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {accuracy}")

# Print probabilities
print("Predicted probabilities for each class:")
print(y_pred_prob)

In [ ]:
#Probability of single instance

In [ ]:
X_single=X_test[[2]]

In [ ]:
y_pred = xgb_clf.predict_proba(X_single)
print(y_pred)

In [ ]:
aa= df.label.unique()

In [ ]:
# Create the DataFrame
label_prob = pd.DataFrame({
    'Label': aa,
    'Probability': y_pred[0]
})

# Display the DataFrame
print(label_prob)